In [1]:
# # The Bayes filter: predict, then update (notebook)
#
# Thrun's door world: the state is the door, open or closed. The robot can push or do nothing. Its sensor reads
# "open" with probability 0.6 when the door is open and 0.2 when it is closed. We check every number of the Note:
# the static door with two readings, the two-step Bayes filter, and the continuous hallway with Gaussian beliefs.

In [2]:
import numpy as np
import plotly.graph_objects as go

STATES = ("open", "closed")
# measurement probability p(z | x): rows = state x, columns = reading z ("sense open", "sense closed")
P_Z = {"open": {"open": 0.6, "closed": 0.4}, "closed": {"open": 0.2, "closed": 0.8}}
# state transition probability p(x_t | u_t, x_t-1): P_X[u][x_prev][x_new]
P_X = {"push": {"open": {"open": 1.0, "closed": 0.0}, "closed": {"open": 0.8, "closed": 0.2}},
       "do nothing": {"open": {"open": 1.0, "closed": 0.0}, "closed": {"open": 0.0, "closed": 1.0}}}

## 1. A static door and two readings (Note, Section 3)

In [3]:
def bayes(prior, z):
    unnorm = {x: P_Z[x][z] * prior[x] for x in STATES}
    total = sum(unnorm.values())                       # p(z | earlier readings): the evidence
    return {x: v / total for x, v in unnorm.items()}, unnorm, total


b = {"open": 0.5, "closed": 0.5}
b1, u1, e1 = bayes(b, "open")
b2, u2, e2 = bayes(b1, "open")
print("after reading 1:", u1, "evidence", e1, "->", b1)
print("after reading 2:", u2, "evidence", e2, "->", b2)
# the two readings are NOT independent unless we know the state
print("p(z2 = open) before any reading:", 0.6 * 0.5 + 0.2 * 0.5)
print("p(z2 = open | z1 = open):", e2)

after reading 1: {'open': 0.3, 'closed': 0.1} evidence 0.4 -> {'open': 0.7499999999999999, 'closed': 0.25}
after reading 2: {'open': 0.4499999999999999, 'closed': 0.05} evidence 0.4999999999999999 -> {'open': 0.9, 'closed': 0.10000000000000003}
p(z2 = open) before any reading: 0.4
p(z2 = open | z1 = open): 0.4999999999999999


## 2. The Bayes filter on the door world (Note, Section 4)

In [4]:
def predict(bel, u):
    return {x: sum(P_X[u][xp][x] * bel[xp] for xp in STATES) for x in STATES}


def update(bel_bar, z):
    unnorm = {x: P_Z[x][z] * bel_bar[x] for x in STATES}
    eta = 1 / sum(unnorm.values())
    return {x: eta * v for x, v in unnorm.items()}, unnorm, eta


bel = {"open": 0.5, "closed": 0.5}
for t, (u, z) in enumerate((("do nothing", "open"), ("push", "open")), start=1):
    bel_bar = predict(bel, u)
    bel, unnorm, eta = update(bel_bar, z)
    print(f"t = {t}: u = {u}, z = {z}")
    print("   predicted belief:", bel_bar)
    print("   before normalising:", unnorm, " eta =", round(eta, 4))
    print("   belief:", {k: round(v, 4) for k, v in bel.items()})

t = 1: u = do nothing, z = open
   predicted belief: {'open': 0.5, 'closed': 0.5}
   before normalising: {'open': 0.3, 'closed': 0.1}  eta = 2.5
   belief: {'open': 0.75, 'closed': 0.25}
t = 2: u = push, z = open
   predicted belief: {'open': 0.95, 'closed': 0.05}
   before normalising: {'open': 0.57, 'closed': 0.010000000000000002}  eta = 1.7241
   belief: {'open': 0.9828, 'closed': 0.0172}


## 3. The same two steps on a continuous position (Note, Section 6)

The robot's position in a straight hallway, on a 1 mm grid. Belief: Gaussian, mean 0 m, standard deviation 0.6 m.
Command: drive 3 m forward, with standard deviation 0.9 m. Reading: 3.6 m, with standard deviation 0.7 m.
The prediction is a sum (an integral) over every previous position; the correction multiplies and normalises.

In [5]:
x = np.arange(-4, 9, 0.01)                                # positions on a 1 cm grid
dx = 0.01


def gauss(x, m, s):
    return np.exp(-0.5 * ((x - m) / s) ** 2) / (s * np.sqrt(2 * np.pi))


def mean_sd(p):
    m = (x * p).sum() * dx
    return m, np.sqrt(((x - m) ** 2 * p).sum() * dx)


prior = gauss(x, 0, 0.6)
# prediction: for every new position, add up (probability of having been at x') x (probability of moving x - x')
MOTION = gauss(x[:, None] - x[None, :], 3, 0.9)           # MOTION[i, j] = p(x_i | drive 3 m, x_j)
predicted = MOTION @ prior * dx
likelihood = gauss(x, 3.6, 0.7)                           # p(z = 3.6 | x)
posterior = likelihood * predicted
posterior /= posterior.sum() * dx
for name, p in (("belief before", prior), ("predicted", predicted), ("reading alone", likelihood),
                ("after the reading", posterior)):
    m, s = mean_sd(p)
    print(f"{name:18s} mean {m:.3f} m, standard deviation {s:.3f} m")
print("variances add:", np.sqrt(0.6 ** 2 + 0.9 ** 2))
print("inverse variances add:", 1 / np.sqrt(1 / (0.6 ** 2 + 0.9 ** 2) + 1 / 0.7 ** 2))

belief before      mean 0.000 m, standard deviation 0.600 m
predicted          mean 3.000 m, standard deviation 1.082 m
reading alone      mean 3.600 m, standard deviation 0.700 m
after the reading  mean 3.423 m, standard deviation 0.588 m
variances add: 1.0816653826391966
inverse variances add: 0.5876746087616279


In [6]:
fig = go.Figure()
for name, p in (("belief before", prior), ("predicted", predicted), ("reading", likelihood),
                ("after the reading", posterior)):
    fig.add_trace(go.Scatter(x=x, y=p, name=name))
fig.update_layout(template="simple_white", xaxis_title="position (m)", yaxis_title="density (per m)",
                  title="Predict widens, correct sharpens")
fig.show()